In [ ]:
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver
!pip install -q selenium pandas

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../0-systemd-sysv_255.4-1ubuntu8.18_amd64.deb ...
Unpacking systemd-sysv (255.4-1ubuntu8.18) over (255.4-1ubuntu8.17) ...
Preparing to unpack .../1-systemd-dev_255.4-1ubuntu8.18_all.deb ...
Unpacking systemd-dev (255.4-1ubuntu8.18) over (255.4-1ubuntu8.17) ...
Preparing to unpack .../2-libpam-systemd_255.4-1ubuntu8.18_amd64.deb ...
Unpacking libpam-systemd:amd64 (255.4-1ubuntu8.18) over (255.4-1ubuntu8.17) ...
Preparing to unpack .../3-libapparmor1_4.0.1really4.0.1-0ubuntu0.24.04.9_amd64.deb ...
Unpacking libapparmor1:amd64 (4.0.1really4.0.1-0ubuntu0.24.04.9) over (4.0.1really4.0.1-0ubuntu0.24.04.8) ...
Preparing to unpack .../4-systemd_255.4-1ubuntu8.18_amd64.deb ...
Unpacking sy

In [ ]:
!apt-get remove -y -qq chromium-chromedriver chromium-browser
!rm -f /usr/bin/chromedriver
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
!pip install -q --upgrade selenium
!google-chrome --version

(Reading database ... 123407 files and directories currently installed.)
Removing chromium-chromedriver (2:1snap1-0ubuntu2) ...
Removing chromium-browser (2:1snap1-0ubuntu2) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
Selecting previously unselected package at-spi2-common.
(Reading database ... 123386 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatk-bridge2.0-0t64:amd64.
Preparing to unpack .../3-libatk-bridge2.0-0t64_2.52.0-1build1_amd64.deb ...
Unp

In [ ]:
import time, random, re, os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options


TEST_MODE = False          
CHECKIN, CHECKOUT = "2027-01-10", "2027-01-15"   

ALL_CITIES = ['Amsterdam', 'Athens', 'Barcelona', 'Berlin', 'Budapest',
              'Lisbon', 'London', 'Paris', 'Rome', 'Vienna']

if TEST_MODE:
    cities, target_count = ['Amsterdam'], 3
else:
    cities, target_count = ALL_CITIES, 50

In [ ]:
def get_details_and_fav(driver, wait):
    rating, reviews, is_fav = "N/A", "0", 0

    try:
        
        try:
            review_link = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//a[contains(@href, 'reviews')] | //button[contains(., 'reviews')]")
            ))
        except:
            
            try:
                h1_parent = driver.find_element(By.TAG_NAME, "h1").find_element(By.XPATH, "./..").text
                if "New" in h1_parent: return "New", "0", 0
            except: pass
            return rating, reviews, is_fav

        
        link_text = review_link.get_attribute("textContent").strip()
        clean_link = " ".join(link_text.split())

        
        parent_text = review_link.find_element(By.XPATH, "./..").get_attribute("textContent").strip()
        clean_parent = " ".join(parent_text.split())

       
        if "Guest favorite" in clean_link or "Guest favorite" in clean_parent:
            is_fav = 1
            raw = review_link.find_element(By.XPATH, "./..").get_attribute("innerText") or ""
            print("DEBUG fav text:", repr(raw))   
            
            lines = [l.strip() for l in raw.split("\n") if l.strip()]
            for l in lines:
                if re.fullmatch(r"\d\.\d{1,2}", l): rating = l
                m = re.fullmatch(r"(\d[\d,]*)(\s*reviews?)?", l, re.IGNORECASE)
                if m and "." not in l and "," not in l and int(m.group(1)) > 5:
                    reviews = m.group(1)

            
            if reviews in ("0", "") or len(reviews) > 5:
                flat = " ".join(raw.split())
                m = re.search(r"(5\.0|\d\.\d{2})(\d+)\s*reviews?", flat, re.IGNORECASE)
                if m:
                    rating, reviews = m.group(1), m.group(2)

        
        else:
            is_fav = 0
            if "New" in clean_parent:
                rating = "New"
            else:
                r_match = re.search(r"(\d\.\d{1,2})", clean_parent)
                if r_match: rating = r_match.group(1)

            rev_match = re.search(r"(\d+)\s*review", clean_link, re.IGNORECASE)
            if not rev_match:
                rev_match = re.search(r"(\d+)\s*review", clean_parent, re.IGNORECASE)
            if rev_match: reviews = rev_match.group(1)

    except: pass
    return rating, reviews, is_fav

In [ ]:
def make_driver():
    opts = Options()
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opts)

def get_price(driver):
    
    sidebar = driver.find_element(
        By.XPATH, "//div[@data-plugin-in-point-id='BOOK_IT_SIDEBAR']").text
    m = re.search(r"[€£$]\s?[\d,\.]+", sidebar)
    return m.group(0) if m else "N/A"

all_data = []

for i, city in enumerate(cities, start=1):
    URL = (f"https://www.airbnb.com/s/{city}/homes?query={city}"
           f"&checkin={CHECKIN}&checkout={CHECKOUT}&adults=2"
           f"&min_bedrooms=2&room_types%5B%5D=Entire%20home%2Fapt")

    driver = make_driver()
    wait = WebDriverWait(driver, 10)
    scraped = 0
    try:
        driver.get(URL)
        while scraped < target_count:
            print(f"\n=== CITY {i}/{len(cities)}: {city} (collected {scraped}) ===")
            try:
                wait.until(EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-testid='card-container']")))
            except Exception:
                print("لم تظهر نتائج البحث"); break

            cards = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='card-container']")
            urls = []
            for c in cards:
                try: urls.append(c.find_element(By.TAG_NAME, "a").get_attribute("href"))
                except Exception: continue

            main_window = driver.current_window_handle
            for url in urls:
                if scraped >= target_count: break
                try:
                    driver.execute_script(f"window.open('{url}', '_blank');")
                    driver.switch_to.window(driver.window_handles[-1])
                    wait.until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

                    name = driver.find_element(By.TAG_NAME, "h1").text

                   
                    try:
                        property_type = driver.find_element(By.TAG_NAME, "h2").text.split(" in ")[0]
                    except Exception:
                        property_type = "N/A"

                    driver.execute_script("window.scrollBy(0, 500);")
                    time.sleep(0.5)
                    body_text = driver.find_element(By.TAG_NAME, "body").text
                    is_superhost = "Yes" if "Superhost" in body_text else "No"

                    price = get_price(driver)
                    rating, reviews, is_fav = get_details_and_fav(driver, wait)

                    all_data.append({
                        "Name": name, "city": city, "Property_type": property_type,
                        "Price": price, "Rating": rating, "Reviews": reviews,
                        "Superhost": is_superhost, "is_fav": is_fav, "URL": url})
                    scraped += 1
                    print(f"[{len(all_data)}] {property_type} | {price} | ★{rating} ({reviews}) | Fav:{is_fav} | SH:{is_superhost}")
                    time.sleep(random.uniform(1.5, 3.0))
                except Exception as e:
                    print("تخطي عقار:", type(e).__name__)
                finally:
                    if len(driver.window_handles) > 1:
                        driver.close()
                        driver.switch_to.window(main_window)

            if scraped < target_count:
                try:
                    nxt = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next']")
                    driver.execute_script("arguments[0].click();", nxt)
                    time.sleep(4)
                except Exception:
                    print("لا توجد صفحة تالية"); break
    finally:
        driver.quit()

print(f"\nانتهى. إجمالي العقارات: {len(all_data)}")


=== CITY 1/10: Amsterdam (collected 0) ===
DEBUG fav text: 'Guest\nfavorite\nOne of the most loved homes on Airbnb, according to guests\nRated 4.91 out of 5 stars.\n4.91\n626 reviews\n626\nReviews'
[1] Entire rental unit | $2,030 | ★4.91 (626) | Fav:1 | SH:Yes

=== CITY 1/10: Amsterdam (collected 1) ===
[2] N/A | $1,210 | ★4.8 (201) | Fav:0 | SH:Yes
DEBUG fav text: 'Guest\nfavorite\nOne of the most loved homes on Airbnb, according to guests\nRated 5.0 out of 5 stars.\n5.0\n18 reviews\n18\nReviews'
[3] N/A | $1,850 | ★5.0 (18) | Fav:1 | SH:No
DEBUG fav text: 'Guest\nfavorite\nOne of the most loved homes on Airbnb, according to guests\nRated 5.0 out of 5 stars.\n5.0\n11 reviews\n11\nReviews'
[4] Entire guesthouse | $1,203 | ★5.0 (11) | Fav:1 | SH:Yes
[5] Boat | $1,903 | ★4.85 (107) | Fav:0 | SH:Yes
[6] Entire condo | $2,072 | ★4.85 (223) | Fav:0 | SH:Yes
DEBUG fav text: 'Guest\nfavorite\nOne of the most loved homes on Airbnb, according to guests\nRated 4.96 out of 5 stars.\n4.96\n84 rev

In [ ]:
os.makedirs("transformed_data", exist_ok=True)
df_scraped = pd.DataFrame(all_data)
df_scraped.to_csv("transformed_data/Airbnb_Scraped_Data.csv", index=False)
df_scraped

,Name,city,Property_type,Price,Rating,Reviews,Superhost,is_fav,URL
0,Amstel Imperial,Amsterdam,Entire rental unit,"$2,030",4.91,626,Yes,1,https://www.airbnb.com/rooms/18059130?adults=2...
1,Peaceful Wooden Guesthouse @ Amsterdam Noord,Amsterdam,Entire guesthouse,"$1,203",5.0,11,Yes,1,https://www.airbnb.com/rooms/17114586420309568...
2,Luxury Amsterdam Houseboat Listing,Amsterdam,Boat,"$1,903",4.85,107,Yes,0,https://www.airbnb.com/rooms/12419811535441422...


In [ ]:
for i, row in df_scraped.iterrows():
    print(i, row["Property_type"], row["Price"], "\n", row["URL"], "\n")

0 Entire rental unit $2,030 
 https://www.airbnb.com/rooms/18059130?adults=2&check_in=2027-01-10&check_out=2027-01-15&search_mode=regular_search&photo_id=290347356&source_impression_id=p3_912cf175-5641-4877-be79-1138a1b5d152_51e3ad35-d961-4f11-90b5-8c73a7049e30_0_18059130_0&previous_page_section_name=1000&federated_search_id=912cf175-5641-4877-be79-1138a1b5d152 

1 Entire guesthouse $1,203 
 https://www.airbnb.com/rooms/1711458642030956897?adults=2&check_in=2027-01-10&check_out=2027-01-15&search_mode=regular_search&photo_id=2698145939&source_impression_id=p3_84f228f1-2f90-4681-bb8f-c40a0a5c3f2d_979c349a-b314-4e42-8a09-d7c3dc91884d_0_1711458642030956897_0&previous_page_section_name=1000&federated_search_id=84f228f1-2f90-4681-bb8f-c40a0a5c3f2d 

2 Boat $1,903 
 https://www.airbnb.com/rooms/1241981153544142223?adults=2&check_in=2027-01-10&check_out=2027-01-15&search_mode=regular_search&photo_id=2017064922&source_impression_id=p3_84f228f1-2f90-4681-bb8f-c40a0a5c3f2d_979c349a-b314-4e42-8a09

In [ ]:
df_scraped["city"].value_counts()